# **FOREWORD**

This is a simple blend kernel that accepts 3 public notebooks as starter kernels - 

1. [LB 0.792](https://www.kaggle.com/code/waterjoe/birdclef2026-submit-baseline)
2. [LB 0.786](https://www.kaggle.com/code/waterjoe/birdclef2026-submit-baseline)
3. [LB 0.749](https://www.kaggle.com/code/ravi20076/birdclef2026-openvino-starter-v1)


I standardize all inferences to OpenVino to maintain consistency. <br>
I also use the same fallback folder to be able to visualize the blend properly, this has no effect on the actual test set though.

All credits to the base work. Best regards!

**My other kernels in the assignment** <br>
1. [Preprocessor](https://www.kaggle.com/code/ravi20076/birdclef2026-preprocessing-v1)
2. [Baseline OpenVino inference](https://www.kaggle.com/code/ravi20076/birdclef2026-openvino-starter-v1)
3. [Supplements](https://www.kaggle.com/code/ravi20076/birdclef2026-supplements-v1)

# **SCRIPT LIBRARY**

These are the individual scrits used for the blend. One may add more scripts to this kernel here / link this with a GitHub repo as well.

## **LB 0.792**

In [ ]:
%%writefile lb792.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import glob
import timm
import torch
import torch.nn as nn
import torchaudio
import torchvision
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm
from torch.utils.data import Dataset

ov_device       = "CPU"
PATH            = '/kaggle/input/competitions/birdclef-2026/'
TEST_PATH       = PATH + 'test_soundscapes/'
TRAIN_PATH      = PATH + 'train_soundscapes/'
N_FALLBACK      = 2
SUBMISSION_FILE = "lb792.csv"
model_dir       = "/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models"

DUR = 5
SR  = 32000

class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512,
                 f_min=20, f_max=16000, channels=1, norm="slaney",
                 mel_scale="htk", target_size=(256, 256), top_db=80.0, **kwargs):
        super().__init__()
        self.channels = channels
        self.top_db   = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, pad_mode="reflect", power=2.0,
            norm=norm, center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin     = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = x.dim() == 1
        if squeeze:
            x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        mel = self.power_to_db(mel)
        mel = mel.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel = self.resize(mel)
        B, C = mel.shape[:2]
        flat = mel.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel  = (mel - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel = mel.squeeze(0)
        return mel

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        cfg = {
            'backbone':        'tf_efficientnetv2_b0',
            'backbone_pooling':'avg',
            'dropout':          0.1,
            'pretrained':       False,
            'channels':         1,
            'num_labels':       234, # Will dynamically update
        }
        if config:
            cfg.update(config)
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=cfg['pretrained'],
            num_classes=cfg['num_labels'],
            global_pool=cfg['backbone_pooling'],
            in_chans=cfg['channels'],
            drop_rate=cfg['dropout'],
        )

    def forward(self, x):
        return self.backbone(x)


class BirdDataset(Dataset):
    def __init__(self, paths, spec_transform):
        self.paths = paths
        self.spec  = spec_transform
        self.chunk_len = SR * DUR
        self.half_chunk = self.chunk_len // 2

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        filepath = self.paths[idx]
        filename = filepath.split('/')[-1].split('.')[0]
        try:
            wav, _ = torchaudio.load(filepath)
            wav = wav.float()[0]  
            
            n_seg = len(wav) // self.chunk_len 
            
            if n_seg == 0:
                wav = torch.nn.functional.pad(wav, (0, self.chunk_len - len(wav)))
                n_seg = 1
            else:
                wav = wav[: n_seg * self.chunk_len]
            
            # 1. Standard Chunks (0-5s, 5-10s, ...)
            wav_std = wav.reshape((n_seg, self.chunk_len))
            
            # 2. Shifted Chunks for TTA (2.5-7.5s, 7.5-12.5s, ...)
            wav_padded = torch.nn.functional.pad(wav, (0, self.half_chunk))
            wav_shift = torch.stack([
                wav_padded[i * self.chunk_len + self.half_chunk : (i+1) * self.chunk_len + self.half_chunk]
                for i in range(n_seg)
            ])
            
            mel_std   = torch.stack([self.spec(wav_std[i]) for i in range(n_seg)])
            mel_shift = torch.stack([self.spec(wav_shift[i]) for i in range(n_seg)])
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names
            
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            n_seg = int(240 / DUR) 
            mel_std   = torch.zeros((n_seg, 1, 256, 256))
            mel_shift = torch.zeros((n_seg, 1, 256, 256))
            names =[f"{filename}_{int((i + 1) * DUR)}" for i in range(n_seg)]
            return mel_std.numpy().astype(np.float32), mel_shift.numpy().astype(np.float32), names

def export_to_openvino(ckpt_path, ov_model_path, config=None):
    if os.path.exists(ov_model_path):
        return
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    
    num_labels = state['backbone.classifier.weight'].shape[0]
    cfg = config.copy() if config else {}
    cfg['num_labels'] = num_labels
    
    model = BirdModel(cfg)
    model.load_state_dict(state)
    model.eval()
    
    dummy    = torch.zeros(1, 1, 256, 256)
    ov_model = ov.convert_model(model, example_input=dummy)
    ov_model.reshape([-1, 1, 256, 256]) # Support dynamic batch sizes
    ov.save_model(ov_model, ov_model_path)

def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    return core.compile_model(ov_model, device_name=ov_device)

def predict_batch(compiled_models, batch_np):
    logits_list =[]
    for compiled in compiled_models:
        logits = compiled(batch_np)[compiled.output(0)]
        logits_list.append(logits)
    
    # 1. Ensemble average the logits directly
    mean_logits = np.mean(logits_list, axis=0)
    
    # 2. Standard sigmoid (removed temperature scaling to preserve native calibration)
    probs = 1.0 / (1.0 + np.exp(np.clip(-mean_logits, -50, 50)))
    return probs

def run_inference(compiled_models, paths, spec):
    dataset  = BirdDataset(paths, spec)
    all_preds, all_names = [],[]

    for mel_std, mel_shift, names in tqdm(dataset, desc="Infer"):
        
        preds_std   = predict_batch(compiled_models, mel_std)
        preds_shift = predict_batch(compiled_models, mel_shift)
        
        # 1. Weave TTA Shifted Predictions (Recovers birds cut in half)
        preds = np.copy(preds_std)
        n = len(preds)
        for i in range(n):
            shift_prev = preds_shift[i-1] if i > 0 else preds_std[i]
            shift_curr = preds_shift[i]
            # 50% Standard Window + 50% overlapping halves
            preds[i] = 0.50 * preds_std[i] + 0.25 * shift_prev + 0.25 * shift_curr
        
        # 2. Wider Gaussian Time-Domain Smoothing
        smoothed_preds = np.copy(preds)
        for i in range(n):
            p_m2 = preds[max(0, i-2)]
            p_m1 = preds[max(0, i-1)]
            p_0  = preds[i]
            p_p1 = preds[min(n-1, i+1)]
            p_p2 = preds[min(n-1, i+2)]
            # Smooth the probabilities out across a 25-second window
            smoothed_preds[i] = 0.05*p_m2 + 0.15*p_m1 + 0.60*p_0 + 0.15*p_p1 + 0.05*p_p2
            
        # 3. Global File Context (The Grandmaster Trick, without clipping!)
        file_max = np.max(smoothed_preds, axis=0)
        
        # Blend: 80% local chunk + 20% global file context
        # This securely boosts the AUC rank of chunks in files where the bird is confirmed to exist
        final_preds = 0.80 * smoothed_preds + 0.20 * file_max
        
        all_preds.append(final_preds)
        all_names.extend(names)

    return np.concatenate(all_preds, axis=0), all_names


def create_submission(preds, names, train_labels):
    sample_sub = pd.read_csv(PATH + 'sample_submission.csv')
    expected_cols = sample_sub.columns.tolist()
    
    # Check if predictions are completely empty
    if len(preds) == 0:
        print("No predictions generated. Saving sample_submission directly.")
        sample_sub.to_csv(SUBMISSION_FILE, index=False)
        return sample_sub
        
    df_preds = pd.DataFrame(preds, columns=train_labels[:preds.shape[1]])
    df_preds['row_id'] = names
    
    final_dict = {'row_id': df_preds['row_id']}
    for col in expected_cols:
        if col == 'row_id': continue
        if col in df_preds.columns:
            final_dict[col] = df_preds[col].values
        else:
            final_dict[col] = 0.0
            
    final_df = pd.DataFrame(final_dict)
    final_df = final_df[expected_cols]
    final_df.to_csv(SUBMISSION_FILE, index=False)
    
    print(f"Submission saved: {final_df.shape}\n\n")
    return final_df

def main():
    train_labels = sorted(pd.read_csv(PATH + 'train.csv')['primary_label'].unique().tolist())
    
    # 1. Search for test files (Using glob to support recursive search)
    paths = glob.glob(TEST_PATH + "**/*.ogg", recursive=True)
    
    if not paths:
        paths = sorted(glob.glob(TRAIN_PATH + "**/*.ogg", recursive=True))[:N_FALLBACK]

    # 3. Handle absolutely no files case elegantly!
    if not paths:
        print(f"Files: 0. Generating default submission directly to avoid crashes.")
        create_submission([],[], train_labels)
        return

    print(f"Files: {len(paths)}, Model Output Classes: {len(train_labels)}")

    spec = Spectrogram(sr=SR, n_fft=2048, n_mels=256, hop_length=512,
                       f_min=20, f_max=16000, channels=1,
                       target_size=(256, 256), top_db=80.0)

    all_ckpts =[os.path.join(model_dir, f) for f in sorted(os.listdir(model_dir)) if f.endswith('.pth')]
    best_ckpts =[p for p in all_ckpts if '_best.pth' in os.path.basename(p)]
    ckpt_paths = best_ckpts if best_ckpts else all_ckpts

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)

    compiled_models =[]
    for ckpt in ckpt_paths:
        ov_name = os.path.splitext(os.path.basename(ckpt))[0] + '.xml'
        ov_path = os.path.join(ov_export_dir, ov_name)
        export_to_openvino(ckpt, ov_path)
        compiled_models.append(compile_ov_model(ov_path))

    print(f"Loaded {len(compiled_models)} OpenVINO model(s)")

    preds, names = run_inference(compiled_models, paths, spec)
    create_submission(preds, names, train_labels)

if __name__ == "__main__":
    main()

## **LB 0.786**

In [ ]:
%%writefile lb786.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as AT
import torchvision.models as models
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm

ov_device       = "CPU"
ROOT            = "/kaggle/input/competitions/birdclef-2026/"
TEST_DIR        = os.path.join(ROOT, "test_soundscapes")
TRAIN_DIR       = os.path.join(ROOT, "train_soundscapes")
MODEL_PATH      = "/kaggle/input/notebooks/waterjoe/birdclef2026-train-baseline/baseline_2026.pth"
SUBMISSION_FILE = "lb786.csv"
N_FALLBACK      = 2

sample_rate  = 32000
wav_sec      = 5
segment_len  = sample_rate * wav_sec

taxonomy     = pd.read_csv(os.path.join(ROOT, "taxonomy.csv"))
CLASS_LABELS = sorted(taxonomy.primary_label.unique())
num_classes  = len(CLASS_LABELS)

print("Classes:", num_classes)

# ==============================
# MEL TRANSFORM (CPU — used for export only)
# ==============================

mel_transform = AT.MelSpectrogram(
    sample_rate=32000, n_fft=1024, win_length=1024, hop_length=512,
    f_min=20, f_max=15000, n_mels=128, power=2.0,
    norm="slaney", mel_scale="htk"
)

def normalize_std(spec, eps=1e-23):
    mean = spec.mean()
    std  = spec.std()
    return (spec - mean) / (std + eps)

# ==============================
# MODEL (PyTorch — used for OpenVINO export only)
# ==============================

class BirdModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet34(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        x = torch.cat([x, x, x], 1)
        return self.model(x)

def export_to_openvino(model_path, ov_model_path):
    if os.path.exists(ov_model_path):
        return
    pt_model = BirdModel()
    pt_model.load_state_dict(torch.load(model_path, map_location="cpu"))
    pt_model.eval()
    dummy    = torch.zeros(1, 1, 128, 313)  # [B, C, n_mels, time_frames]
    ov_model = ov.convert_model(pt_model, example_input=dummy)
    ov_model.reshape([-1, 1, 128, 313])
    ov.save_model(ov_model, ov_model_path)
    print(f"Exported OpenVINO model: {ov_model_path}")

def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    return core.compile_model(ov_model, device_name=ov_device)

# ==============================
# AUDIO CHUNKING + TTA
# ==============================

def make_chunks(wav):
    chunks = []
    for i in range(12):
        s     = i * segment_len
        chunk = wav[:, s:s + segment_len]
        if chunk.shape[1] < segment_len:
            chunk = torch.nn.functional.pad(chunk, (0, segment_len - chunk.shape[1]))
        chunks.append(chunk)
    return chunks

def make_tta_chunks(wav):
    shift = segment_len // 2
    wav   = torch.nn.functional.pad(wav, (shift, shift))
    chunks = []
    for i in range(12):
        s = i * segment_len
        chunks.append(wav[:, s:s + segment_len])
    return chunks

def wav_to_spec(chunk):
    spec = mel_transform(chunk)
    spec = torch.log(spec + 1e-10)
    spec = normalize_std(spec)
    return spec  # (1, n_mels, time)

# ==============================
# TEMPORAL SMOOTH
# ==============================

def temporal_smooth(pred):
    smoothed = pred.copy()
    for i in range(len(pred)):
        p_prev = pred[max(i - 1, 0)]
        p_next = pred[min(i + 1, len(pred) - 1)]
        smoothed[i] = 0.6 * pred[i] + 0.2 * p_prev + 0.2 * p_next
    return smoothed

# ==============================
# INFERENCE
# ==============================

def predict_batch(compiled_model, specs_np):
    logits = compiled_model(specs_np)[compiled_model.output(0)]
    return 1.0 / (1.0 + np.exp(np.clip(-logits, -50, 50)))

def predict_file(path, compiled_model):
    wav, _ = torchaudio.load(path)
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    wav = wav / (wav.abs().max() + 1e-7)

    chunks1 = make_chunks(wav)
    chunks2 = make_tta_chunks(wav)

    specs1 = torch.stack([wav_to_spec(c) for c in chunks1]).numpy().astype(np.float32)  # (12, 1, n_mels, T)
    specs2 = torch.stack([wav_to_spec(c) for c in chunks2]).numpy().astype(np.float32)

    pred1 = predict_batch(compiled_model, specs1)
    pred2 = predict_batch(compiled_model, specs2)
    pred  = (pred1 + pred2) / 2
    pred  = temporal_smooth(pred)
    return pred

# ==============================
# MAIN
# ==============================

def main():
    files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(".ogg")]) \
            if os.path.exists(TEST_DIR) else []

    if not files:
        print(f"No ogg files found in test_soundscapes, falling back to train_soundscapes...")
        if os.path.exists(TRAIN_DIR):
            files    = sorted([f for f in os.listdir(TRAIN_DIR) if f.endswith(".ogg")])[:N_FALLBACK]
            audio_dir = TRAIN_DIR
        else:
            print("train_soundscapes not found. Saving empty submission.")
            pd.DataFrame(columns=["row_id"] + CLASS_LABELS).to_csv(SUBMISSION_FILE, index=False)
            return
    else:
        audio_dir = TEST_DIR

    print(f"Files: {len(files)}, Classes: {num_classes}, Device: {ov_device}")

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)
    ov_path = os.path.join(ov_export_dir, "baseline_2026.xml")

    export_to_openvino(MODEL_PATH, ov_path)
    compiled_model = compile_ov_model(ov_path)
    print("OpenVINO model loaded")

    rows, predictions = [], []

    for f in tqdm(files, desc="Infer"):
        path = os.path.join(audio_dir, f)
        pred = predict_file(path, compiled_model)
        base = f.replace(".ogg", "")
        for i in range(len(pred)):
            rows.append(f"{base}_{(i + 1) * 5}")
        predictions.append(pred)

    if predictions:
        predictions = np.vstack(predictions)
        submission  = pd.DataFrame(predictions, columns=CLASS_LABELS)
        submission.insert(0, "row_id", rows)
    else:
        print("No predictions generated. Saving empty submission")
        submission = pd.DataFrame(columns=["row_id"] + CLASS_LABELS)

    submission.to_csv(SUBMISSION_FILE, index=False)
    print(f"Submission saved: {submission.shape}\n\n")

if __name__ == "__main__":
    main()


## **LB 0.749**

In [ ]:
%%writefile lb749.py

from warnings import filterwarnings
filterwarnings("ignore")

import os
import math
import timm
import torch
import torch.nn as nn
import torchaudio
import torchvision
import numpy as np
import pandas as pd
import openvino as ov
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from torch.utils.data import Dataset, DataLoader

batch_size      = 64
ov_device       = "CPU"
PATH            = '/kaggle/input/competitions/birdclef-2026/'
TEST_PATH       = PATH + 'test_soundscapes/'
TRAIN_PATH      = PATH + 'train_soundscapes/'
N_FALLBACK      = 2
taxonomy_csv    = PATH + 'taxonomy.csv'
SUBMISSION_FILE = "lb749.csv"
model_dir       = "/kaggle/input/notebooks/antoinemasq/birdclef-2026-pytorch-baseline-training/models"

DUR  = 5
SR   = 32000

class Spectrogram(nn.Module):
    def __init__(self, sr=32000, n_fft=2048, n_mels=256, hop_length=512,
                 f_min=20, f_max=16000, channels=1, norm="slaney",
                 mel_scale="htk", target_size=(256, 256), top_db=80.0, **kwargs):
        super().__init__()
        self.channels = channels
        self.top_db   = top_db
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            n_mels=n_mels, f_min=f_min, f_max=f_max,
            mel_scale=mel_scale, pad_mode="reflect", power=2.0,
            norm=norm, center=True,
        )
        self.resize = torchvision.transforms.Resize(size=target_size)

    def power_to_db(self, S):
        amin    = 1e-10
        log_spec = 10.0 * torch.log10(S.clamp(min=amin))
        log_spec -= 10.0 * torch.log10(torch.tensor(amin).to(S))
        if self.top_db is not None:
            max_val  = log_spec.flatten(-2).max(dim=-1).values[..., None, None]
            log_spec = torch.maximum(log_spec, max_val - self.top_db)
        return log_spec

    def forward(self, x, resize=True):
        squeeze = x.dim() == 1
        if squeeze:
            x = x.unsqueeze(0)
        mel = self.mel_transform(x)
        mel = self.power_to_db(mel)
        mel = mel.unsqueeze(1).repeat(1, self.channels, 1, 1)
        if resize:
            mel = self.resize(mel)
        B, C = mel.shape[:2]
        flat = mel.view(B, C, -1)
        mins = flat.min(dim=-1).values[..., None, None]
        maxs = flat.max(dim=-1).values[..., None, None]
        mel  = (mel - mins) / (maxs - mins + 1e-7)
        if squeeze:
            mel = mel.squeeze(0)
        return mel

class BirdModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        cfg = {
            'backbone':        'tf_efficientnetv2_b0',
            'backbone_pooling':'avg',
            'dropout':          0.1,
            'pretrained':       False,
            'channels':         1,
            'num_labels':       234,
        }
        if config:
            cfg.update(config)
        self.backbone = timm.create_model(
            cfg['backbone'],
            pretrained=cfg['pretrained'],
            num_classes=cfg['num_labels'],
            global_pool=cfg['backbone_pooling'],
            in_chans=cfg['channels'],
            drop_rate=cfg['dropout'],
        )

    def forward(self, x):
        return self.backbone(x)

class BirdDataset(Dataset):
    def __init__(self, paths, spec_transform):
        self.paths = paths
        self.spec  = spec_transform
        self.n_seg = int(60 / DUR)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        filepath = self.paths[idx]
        try:
            wav, _  = torchaudio.load(filepath)
            wav     = wav.float()[:, :SR * 60]
            wav     = wav.reshape((self.n_seg, SR * DUR))
            mel     = torch.stack([self.spec(wav[i]) for i in range(len(wav))])
            names   = [
                filepath.split('/')[-1].split('.')[0] + '_' + str(i * DUR + DUR)
                for i in range(self.n_seg)
            ]
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            mel   = torch.zeros((self.n_seg, 1, 256, 256))
            names = [filepath.split('/')[-1].split('.')[0] + '_' + str(i * DUR + DUR)
                     for i in range(self.n_seg)]
        return mel.numpy().astype(np.float32), names   # (n_seg, C, H, W)

def export_to_openvino(ckpt_path, ov_model_path, config=None):
    if os.path.exists(ov_model_path):
        return
    print(f"Exporting {ckpt_path} -> {ov_model_path}")
    state = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        state = state['model_state_dict']
    # Infer num_labels directly from checkpoint to avoid size mismatch
    num_labels = state['backbone.classifier.weight'].shape[0]
    cfg = config.copy() if config else {}
    cfg['num_labels'] = num_labels
    model = BirdModel(cfg)
    model.load_state_dict(state)
    model.eval()
    dummy    = torch.zeros(1, 1, 256, 256)
    ov_model = ov.convert_model(model, example_input=dummy)
    ov.save_model(ov_model, ov_model_path)
    print(f"Saved: {ov_model_path}")


def compile_ov_model(ov_model_path):
    core     = ov.Core()
    ov_model = core.read_model(ov_model_path)
    compiled = core.compile_model(ov_model, device_name=ov_device)
    return compiled

def predict_batch(compiled_models, batch_np):
    """batch_np: (B, C, H, W)"""
    preds = []
    for compiled in compiled_models:
        logits = compiled.infer_new_request({0: batch_np})[compiled.output(0)]
        probs  = 1.0 / (1.0 + np.exp(-logits))
        preds.append(probs)
    return np.mean(preds, axis=0)


def run_inference(compiled_models, paths, spec, num_labels, labels):
    dataset  = BirdDataset(paths, spec)
    all_preds, all_names = [], []

    for mel_segs, names in tqdm(dataset, desc="Infer"):
        # mel_segs: (n_seg, C, H, W)
        preds = predict_batch(compiled_models, mel_segs)   # (n_seg, num_labels)
        all_preds.append(preds)
        all_names.extend(names)

    return np.concatenate(all_preds, axis=0), all_names

def create_submission(preds, names, all_labels, train_labels):
    n_pred_cols = preds.shape[1]
    # train_labels may be a subset; use only as many as the model predicted
    used_labels = train_labels[:n_pred_cols]
    df = pd.DataFrame(np.zeros((len(preds), len(all_labels))), columns=all_labels)
    df[used_labels] = preds
    df.insert(0, 'row_id', names)
    df.to_csv(SUBMISSION_FILE, index=False)

    print(f"Submission saved: {df.shape}\n\n")
    return df

def main():
    taxonomy_df  = pd.read_csv(taxonomy_csv)
    all_labels   = sorted(taxonomy_df['primary_label'].unique().tolist())
    train_labels = sorted(pd.read_csv(PATH + 'train.csv')['primary_label'].unique().tolist())
    num_labels   = len(train_labels)

    paths = [TEST_PATH + x for x in os.listdir(TEST_PATH) if x.endswith('.ogg')]
    if not paths:
        paths = sorted([TRAIN_PATH + x for x in os.listdir(TRAIN_PATH) if x.endswith('.ogg')])[:N_FALLBACK]

    print(f"Files: {len(paths)}, Classes: {num_labels}, Device: {ov_device}")

    spec = Spectrogram(sr=SR, n_fft=2048, n_mels=256, hop_length=512,
                       f_min=20, f_max=16000, channels=1,
                       target_size=(256, 256), top_db=80.0)

    ckpt_paths = [os.path.join(model_dir, f)
                  for f in os.listdir(model_dir) if f.endswith('.pth')]

    ov_export_dir = "/kaggle/working/ov_models"
    os.makedirs(ov_export_dir, exist_ok=True)

    compiled_models = []
    for ckpt in ckpt_paths:
        ov_name = os.path.splitext(os.path.basename(ckpt))[0] + '.xml'
        ov_path = os.path.join(ov_export_dir, ov_name)
        export_to_openvino(ckpt, ov_path)
        compiled_models.append(compile_ov_model(ov_path))

    print(f"Loaded {len(compiled_models)} OpenVINO model(s)")

    preds, names = run_inference(compiled_models, paths, spec, num_labels, train_labels)
    create_submission(preds, names, all_labels, train_labels)


if __name__ == "__main__":
    main()

## **BLEND**

In [ ]:
%%writefile blend.py

import pandas as pd, os
from warnings import filterwarnings
filterwarnings("ignore")

print(f"\n---> Starting model blend")

sub792 = pd.read_csv(f"lb792.csv", index_col = "row_id")
sub786 = pd.read_csv(f"lb786.csv", index_col = "row_id")[sub792.columns]
sub749 = pd.read_csv(f"lb749.csv", index_col = "row_id")[sub792.columns]

sub_fl = sub792 * 0.60 + sub786 * 0.30 + sub749 * 0.10

print(f"---> Blended submission file shape = {sub_fl.shape}\n")
sub_fl.to_csv(f"submission.csv", index = True)

# **SUBMISSION**

In [ ]:
import pandas as pd, os, time, sys

!python lb792.py
!python lb786.py
!python lb749.py
!python blend.py

print()
display(
    pd.read_csv(
        "submission.csv", index_col = "row_id"
    ).head(5)
)

print()
!ls